In [1]:
import os
import numpy as np
import pandas as pd
import scanpy as sc
import anndata as an
import sys
sys.path.append('/home/jovyan/notebooks_bidossessi/Inflammatory/Remapped/autoimmuneSC/AIID_notebooks/')

from inflapy.immunoFunc import *

In [2]:
path = "/nfs/team205/bh14/Datasets/RawData/raw_and_AnnData/annDataModified/"

s_path = "/nfs/team205/bh14/Datasets/Remapped/raw_adata/ummapped/"

#### Perez dataset

In [3]:
adata = sc.read_h5ad(path+"Perez_2022_modified.h5ad", backed="r")

In [4]:
adata

AnnData object with n_obs × n_vars = 1263676 × 30933 backed at '/nfs/team205/bh14/Datasets/RawData/raw_and_AnnData/annDataModified/Perez_2022_modified.h5ad'
    obs: 'library_uuid', 'assay_ontology_term_id', 'mapped_reference_annotation', 'is_primary_data', 'cell_type_ontology_term_id', 'author_cell_type', 'cell_state', 'author_cluster', 'sample_uuid', 'tissue_ontology_term_id', 'development_stage_ontology_term_id', 'disease_state', 'suspension_enriched_cell_types', 'suspension_uuid', 'suspension_type', 'donor_uuid', 'ethnicity_ontology_term_id', 'organism_ontology_term_id', 'disease_ontology_term_id', 'sex_ontology_term_id', 'Processing_Cohort', 'ct_cov', 'ind_cov', 'cell_type', 'assay', 'disease', 'organism', 'sex', 'tissue', 'ethnicity', 'development_stage', 'datasets_id'
    var: 'feature_biotype', 'feature_is_filtered', 'feature_name', 'feature_reference'
    uns: 'X_normalization', 'default_embedding', 'layer_descriptions', 'schema_version', 'title'
    obsm: 'X_umap'

In [5]:
adata.obs.disease_state.value_counts()

managed    696626
na         486418
flare       55120
treated     25512
Name: disease_state, dtype: int64

In [6]:
adata = adata[adata.obs.disease_state != "managed"]

In [9]:
adata.obs.donor_uuid.value_counts()

d6b9e3a7-9f15-4931-a07d-4f4a03991d68    13543
1d15a70f-8b4c-4d49-a08b-c7098cd3b170    12768
79dd16e4-4d95-445f-a2b8-2d2e10872774    12491
4ea8244c-06f7-49b6-b246-63195f85a392    11178
03b233c8-d3e2-4438-8b04-0b6621fb5635    10382
                                        ...  
e3d72ad1-03a0-4358-9dc6-cec4b8018512     1899
1bed79e5-39d1-461e-a189-7a70e95b250d     1810
2a3821dd-89d1-4f22-b36e-6d2630e71e16     1487
88e90be3-39e5-4a3b-9218-6f849e4dc8db     1189
94ac9d2f-36bb-4998-8213-dde641efebb1      456
Name: donor_uuid, Length: 118, dtype: int64

In [10]:
adata = adata.to_memory()

In [11]:
adata

AnnData object with n_obs × n_vars = 567050 × 30933
    obs: 'library_uuid', 'assay_ontology_term_id', 'mapped_reference_annotation', 'is_primary_data', 'cell_type_ontology_term_id', 'author_cell_type', 'cell_state', 'author_cluster', 'sample_uuid', 'tissue_ontology_term_id', 'development_stage_ontology_term_id', 'disease_state', 'suspension_enriched_cell_types', 'suspension_uuid', 'suspension_type', 'donor_uuid', 'ethnicity_ontology_term_id', 'organism_ontology_term_id', 'disease_ontology_term_id', 'sex_ontology_term_id', 'Processing_Cohort', 'ct_cov', 'ind_cov', 'cell_type', 'assay', 'disease', 'organism', 'sex', 'tissue', 'ethnicity', 'development_stage', 'datasets_id'
    var: 'feature_biotype', 'feature_is_filtered', 'feature_name', 'feature_reference'
    uns: 'X_normalization', 'default_embedding', 'layer_descriptions', 'schema_version', 'title'
    obsm: 'X_umap'

In [12]:
adata.obs.datasets_id.value_counts()

Perez_2022    567050
Name: datasets_id, dtype: int64

In [13]:
adata.obs = adata.obs[['disease', 'sex', 'tissue', 'development_stage',
                     'author_cell_type', 'disease_state', 'donor_uuid']]

In [14]:
adata.obs.rename(columns={'author_cell_type':'author_annotation', 'donor_uuid': 'donor_id'},
                inplace=True)

In [15]:
adata.obs

,disease,sex,tissue,development_stage,author_annotation,disease_state,donor_id
index,,,,,,,
CAAGGCCAGTATCGAA-1-1-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-1-0-0-0-0-0,Control,female,blood,28-year-old human stage,T4,na,abdb21c1-f9a6-4546-b7cc-52c6fb485e4c
AAGTCTGGTCTACCTC-1-1-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0,Systemic lupus erythematosus,female,blood,34-year-old human stage,cM,flare,8a10ad3a-1725-4df3-a1bb-2d00b0712b92
GAACATCCAGCTATTG-1-1-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-1-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0,Control,female,blood,33-year-old human stage,T4,na,ebd37d14-1745-43b6-971f-e1e9843a258b
TACCTATTCTACTATC-1-1-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-1-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0,Control,female,blood,30-year-old human stage,B,na,167a97cf-5fa3-469e-b7a1-1230747a4242
GTCATTTCAGAGTGTG-1-1-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-1-0-0-0-0-0-0,Control,female,blood,68-year-old human stage,T4,na,098af13e-2d39-4ac7-ac3b-c6b45f5b30b7
...,...,...,...,...,...,...,...
ATCTACTAGGAGTTGC-1-1-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-1-0-0-0,Control,female,blood,33-year-old human stage,T4,na,f8453961-779f-4dcc-97c6-c187b5d91f7c
GCATGTACATCGGAAG-1-1-0-1-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0,Systemic lupus erythematosus,male,blood,35-year-old human stage,NK,flare,81908f0d-495c-4f6a-881f-7f4c73251c86
GAATGAACACCGGAAA-1-1-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-1-0-0-0-0-0,Control,female,blood,49-year-old human stage,ncM,na,6427dfec-8320-41de-8bc7-7256c7282d26


In [16]:
adata.obs.replace(['female', 'male', 'blood', 'Control'], ['Female', 'Male', 'Blood', 'Healthy'], inplace=True)

In [17]:
adata.obs["dataset_id"] = np.repeat("Perez_2022", adata.n_obs)

In [18]:
adata.obs["dataset_id"].value_counts()

Perez_2022    567050
Name: dataset_id, dtype: int64

In [19]:
adata.obs["cell_source"] = np.repeat("PBMC", adata.n_obs)


In [20]:
adata.obs['age'] = adata.obs.development_stage.str.removesuffix("-year-old human stage")

In [21]:
adata.obs.development_stage = np.repeat("Adult", adata.n_obs)

In [22]:
adata.obs

,disease,sex,tissue,development_stage,author_annotation,disease_state,donor_id,dataset_id,cell_source,age
index,,,,,,,,,,
CAAGGCCAGTATCGAA-1-1-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-1-0-0-0-0-0,Healthy,Female,Blood,Adult,T4,na,abdb21c1-f9a6-4546-b7cc-52c6fb485e4c,Perez_2022,PBMC,28
AAGTCTGGTCTACCTC-1-1-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0,Systemic lupus erythematosus,Female,Blood,Adult,cM,flare,8a10ad3a-1725-4df3-a1bb-2d00b0712b92,Perez_2022,PBMC,34
GAACATCCAGCTATTG-1-1-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-1-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0,Healthy,Female,Blood,Adult,T4,na,ebd37d14-1745-43b6-971f-e1e9843a258b,Perez_2022,PBMC,33
TACCTATTCTACTATC-1-1-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-1-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0,Healthy,Female,Blood,Adult,B,na,167a97cf-5fa3-469e-b7a1-1230747a4242,Perez_2022,PBMC,30
GTCATTTCAGAGTGTG-1-1-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-1-0-0-0-0-0-0,Healthy,Female,Blood,Adult,T4,na,098af13e-2d39-4ac7-ac3b-c6b45f5b30b7,Perez_2022,PBMC,68
...,...,...,...,...,...,...,...,...,...,...
ATCTACTAGGAGTTGC-1-1-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-1-0-0-0,Healthy,Female,Blood,Adult,T4,na,f8453961-779f-4dcc-97c6-c187b5d91f7c,Perez_2022,PBMC,33
GCATGTACATCGGAAG-1-1-0-1-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0,Systemic lupus erythematosus,Male,Blood,Adult,NK,flare,81908f0d-495c-4f6a-881f-7f4c73251c86,Perez_2022,PBMC,35
GAATGAACACCGGAAA-1-1-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-1-0-0-0-0-0,Healthy,Female,Blood,Adult,ncM,na,6427dfec-8320-41de-8bc7-7256c7282d26,Perez_2022,PBMC,49


In [25]:
adata.write_h5ad(s_path+"PBMC/Perez_2022_flare_treated_Healthy.h5ad")

In [26]:
gc.collect()

332

In [27]:
### Quality control

In [3]:
adata = sc.read_h5ad(s_path+"PBMC/Perez_2022_flare_treated_Healthy.h5ad")

In [4]:
adata.var

,feature_biotype,feature_is_filtered,feature_name,feature_reference
ENSG00000243485,gene,True,MIR1302-2HG,NCBITaxon:9606
ENSG00000237613,gene,True,FAM138A,NCBITaxon:9606
ENSG00000186092,gene,True,OR4F5,NCBITaxon:9606
ENSG00000238009,gene,True,RP11-34P13.7,NCBITaxon:9606
ENSG00000239945,gene,True,RP11-34P13.8,NCBITaxon:9606
...,...,...,...,...
ENSG00000212907,gene,True,MT-ND4L,NCBITaxon:9606
ENSG00000198886,gene,True,MT-ND4,NCBITaxon:9606
ENSG00000198786,gene,True,MT-ND5,NCBITaxon:9606
ENSG00000198695,gene,False,MT-ND6,NCBITaxon:9606


In [5]:
adata.var

,feature_biotype,feature_is_filtered,feature_name,feature_reference
ENSG00000243485,gene,True,MIR1302-2HG,NCBITaxon:9606
ENSG00000237613,gene,True,FAM138A,NCBITaxon:9606
ENSG00000186092,gene,True,OR4F5,NCBITaxon:9606
ENSG00000238009,gene,True,RP11-34P13.7,NCBITaxon:9606
ENSG00000239945,gene,True,RP11-34P13.8,NCBITaxon:9606
...,...,...,...,...
ENSG00000212907,gene,True,MT-ND4L,NCBITaxon:9606
ENSG00000198886,gene,True,MT-ND4,NCBITaxon:9606
ENSG00000198786,gene,True,MT-ND5,NCBITaxon:9606
ENSG00000198695,gene,False,MT-ND6,NCBITaxon:9606


In [6]:
var_df = adata.var

In [7]:
var_df

,feature_biotype,feature_is_filtered,feature_name,feature_reference
ENSG00000243485,gene,True,MIR1302-2HG,NCBITaxon:9606
ENSG00000237613,gene,True,FAM138A,NCBITaxon:9606
ENSG00000186092,gene,True,OR4F5,NCBITaxon:9606
ENSG00000238009,gene,True,RP11-34P13.7,NCBITaxon:9606
ENSG00000239945,gene,True,RP11-34P13.8,NCBITaxon:9606
...,...,...,...,...
ENSG00000212907,gene,True,MT-ND4L,NCBITaxon:9606
ENSG00000198886,gene,True,MT-ND4,NCBITaxon:9606
ENSG00000198786,gene,True,MT-ND5,NCBITaxon:9606
ENSG00000198695,gene,False,MT-ND6,NCBITaxon:9606


In [8]:
print(adata.raw.X)

  (0, 50)	1.0
  (0, 55)	1.0
  (0, 149)	7.0
  (0, 184)	1.0
  (0, 382)	1.0
  (0, 412)	1.0
  (0, 455)	12.0
  (0, 480)	2.0
  (0, 487)	1.0
  (0, 506)	1.0
  (0, 520)	5.0
  (0, 526)	1.0
  (0, 568)	1.0
  (0, 569)	1.0
  (0, 574)	1.0
  (0, 601)	3.0
  (0, 627)	1.0
  (0, 639)	1.0
  (0, 640)	2.0
  (0, 714)	1.0
  (0, 749)	2.0
  (0, 759)	1.0
  (0, 807)	1.0
  (0, 856)	19.0
  (0, 912)	1.0
  :	:
  (567049, 30364)	1.0
  (567049, 30394)	1.0
  (567049, 30441)	1.0
  (567049, 30450)	1.0
  (567049, 30470)	4.0
  (567049, 30471)	1.0
  (567049, 30496)	1.0
  (567049, 30598)	1.0
  (567049, 30685)	1.0
  (567049, 30690)	1.0
  (567049, 30810)	1.0
  (567049, 30822)	1.0
  (567049, 30829)	1.0
  (567049, 30844)	1.0
  (567049, 30875)	1.0
  (567049, 30920)	5.0
  (567049, 30921)	4.0
  (567049, 30922)	16.0
  (567049, 30923)	14.0
  (567049, 30925)	5.0
  (567049, 30926)	8.0
  (567049, 30927)	6.0
  (567049, 30929)	9.0
  (567049, 30930)	1.0
  (567049, 30932)	3.0


In [9]:
adata = adata.raw.to_adata()

In [10]:
adata.var = var_df

In [11]:
adata.var

,feature_biotype,feature_is_filtered,feature_name,feature_reference
ENSG00000243485,gene,True,MIR1302-2HG,NCBITaxon:9606
ENSG00000237613,gene,True,FAM138A,NCBITaxon:9606
ENSG00000186092,gene,True,OR4F5,NCBITaxon:9606
ENSG00000238009,gene,True,RP11-34P13.7,NCBITaxon:9606
ENSG00000239945,gene,True,RP11-34P13.8,NCBITaxon:9606
...,...,...,...,...
ENSG00000212907,gene,True,MT-ND4L,NCBITaxon:9606
ENSG00000198886,gene,True,MT-ND4,NCBITaxon:9606
ENSG00000198786,gene,True,MT-ND5,NCBITaxon:9606
ENSG00000198695,gene,False,MT-ND6,NCBITaxon:9606


In [12]:
adata.obs

,disease,sex,tissue,development_stage,author_annotation,disease_state,donor_id,dataset_id,cell_source,age
index,,,,,,,,,,
CAAGGCCAGTATCGAA-1-1-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-1-0-0-0-0-0,Healthy,Female,Blood,Adult,T4,na,abdb21c1-f9a6-4546-b7cc-52c6fb485e4c,Perez_2022,PBMC,28
AAGTCTGGTCTACCTC-1-1-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0,Systemic lupus erythematosus,Female,Blood,Adult,cM,flare,8a10ad3a-1725-4df3-a1bb-2d00b0712b92,Perez_2022,PBMC,34
GAACATCCAGCTATTG-1-1-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-1-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0,Healthy,Female,Blood,Adult,T4,na,ebd37d14-1745-43b6-971f-e1e9843a258b,Perez_2022,PBMC,33
TACCTATTCTACTATC-1-1-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-1-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0,Healthy,Female,Blood,Adult,B,na,167a97cf-5fa3-469e-b7a1-1230747a4242,Perez_2022,PBMC,30
GTCATTTCAGAGTGTG-1-1-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-1-0-0-0-0-0-0,Healthy,Female,Blood,Adult,T4,na,098af13e-2d39-4ac7-ac3b-c6b45f5b30b7,Perez_2022,PBMC,68
...,...,...,...,...,...,...,...,...,...,...
ATCTACTAGGAGTTGC-1-1-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-1-0-0-0,Healthy,Female,Blood,Adult,T4,na,f8453961-779f-4dcc-97c6-c187b5d91f7c,Perez_2022,PBMC,33
GCATGTACATCGGAAG-1-1-0-1-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0,Systemic lupus erythematosus,Male,Blood,Adult,NK,flare,81908f0d-495c-4f6a-881f-7f4c73251c86,Perez_2022,PBMC,35
GAATGAACACCGGAAA-1-1-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-1-0-0-0-0-0,Healthy,Female,Blood,Adult,ncM,na,6427dfec-8320-41de-8bc7-7256c7282d26,Perez_2022,PBMC,49


In [13]:
gc.collect()

147

In [ ]:
adataQc(adata=adata, path=s_path+"PBMC/QC_Doublet/", file="Perez_2022_flare_treated_Healthy.h5ad")

#####################################################
###                      Perez_2022_flare_treated_Healthy.h5ad                 ###
######################################################
Preprocessing...
Simulating doublets...


In [5]:
adata.obs.donor_id.value_counts()

d6b9e3a7-9f15-4931-a07d-4f4a03991d68    13543
1d15a70f-8b4c-4d49-a08b-c7098cd3b170    12768
79dd16e4-4d95-445f-a2b8-2d2e10872774    12491
4ea8244c-06f7-49b6-b246-63195f85a392    11178
03b233c8-d3e2-4438-8b04-0b6621fb5635    10382
                                        ...  
e3d72ad1-03a0-4358-9dc6-cec4b8018512     1899
1bed79e5-39d1-461e-a189-7a70e95b250d     1810
2a3821dd-89d1-4f22-b36e-6d2630e71e16     1487
88e90be3-39e5-4a3b-9218-6f849e4dc8db     1189
94ac9d2f-36bb-4998-8213-dde641efebb1      456
Name: donor_id, Length: 118, dtype: int64